In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prod","qa"],label = "select Environment")

In [0]:
env = dbutils.widgets.get("environment")
print(env)
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawInvoice"
# print(brznTablName)
silverTablName = f"saleslake_{env}.silver_{env}.cleanedInvoice"
# print(brznTablName)

In [0]:
spark.sql(f"""
INSERT INTO {silverTablName}
SELECT DISTINCT
    TRIM(invoice_id)                                              AS invoice_id,
    TRIM(invoice_number)                                          AS invoice_number,
    CAST(TRIM(customer_id) AS BIGINT)                             AS customer_id,
    TO_DATE(TRIM(invoice_date),  'yyyy-MM-dd')                    AS invoice_date,
    TO_DATE(TRIM(due_date),      'yyyy-MM-dd')                    AS due_date,
    CAST(TRIM(subtotal_amount) AS DECIMAL(18,2))                  AS subtotal_amount,
    UPPER(NULLIF(TRIM(discount_code), ''))                        AS discount_code,
    CAST(TRIM(discount_amount) AS DECIMAL(18,2))                  AS discount_amount,
    CAST(TRIM(tax_amount)      AS DECIMAL(18,2))                  AS tax_amount,
    CAST(TRIM(total_amount)    AS DECIMAL(18,2))                  AS total_amount,
    UPPER(TRIM(payment_status))                                   AS payment_status,
    UPPER(NULLIF(TRIM(payment_method), ''))                       AS payment_method,
    CASE
        WHEN NULLIF(TRIM(payment_date), '') IS NULL THEN NULL
        ELSE TO_DATE(TRIM(payment_date), 'yyyy-MM-dd')
    END                                                           AS payment_date,
    UPPER(TRIM(currency))                                         AS currency,
    UPPER(TRIM(region))                                           AS region,
    UPPER(TRIM(store_id))                                         AS store_id,
    UPPER(TRIM(channel))                                          AS channel,
    LOWER(TRIM(created_by))                                       AS created_by,
    CURRENT_TIMESTAMP()                                           AS ingest_ts
FROM {bronzeTablName}
WHERE ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP('1990-01-01','yyyy-MM-dd'))
    FROM {silverTablName}
)
ORDER BY TRIM(invoice_id)
""")

In [0]:
%sql
SELECT * FROM saleslake_dev.bronze_dev.rawInvoice;

In [0]:
%sql
SELECT * FROM saleslake_dev.silver_dev.cleanedInvoice;